# LLM reasoning experiments on Kaggle (free tier)

Run the cells **in order**. **Keep the accelerator OFF until cell 8** — cells 1–7 are CPU-only validation (environment check, mock smoke, unit tests, optional tiny model). Cell 8 is the first GPU spend.

## Budget discipline: read this before running anything

| Limit | Value | Consequence for this project |
|---|---|---|
| Session runtime | **12 h** hard kill (CPU/GPU) | The runner defaults to a **8.0 h** budget and stops cleanly before the kill. |
| Weekly accelerator quota | **~30 GPU-hours**, resets weekly | ~3 full sessions per week. A wasted session is 1/3 of your week. |
| Accelerator | 2x T4 16GB (Turing, **no bf16**) or 1x P100 16GB | Default dtype is **fp16**. A bf16 request is downgraded automatically. |
| RAM / disk | 29 GB RAM, **20 GB** auto-saved `/kaggle/working` | Zip results; do not leave model weights in `/kaggle/working`. |
| Internet | may be on or off | Prefetch weights+data into a Kaggle Dataset, then run with `HF_HUB_OFFLINE=1`. |

**The three rules that save quota:**
1. Always run the mock smoke test (cell 6) first. If the harness is broken, find out in 10 seconds, not after 40 GPU-minutes.
2. Never re-run inference to fix grading. Grading is a separate cached pass (cell 9) over the raw generations.
3. Never lose a partial run. Results are append-only and keyed by a deterministic id, so re-running the same command **resumes**. See cell 11 for how to carry a run across sessions.

## How resuming works (in one paragraph)

Each example's result is keyed by `uid = hash(model, strategy, dataset, example index, seed, config hash)` and appended to `results.jsonl`, fsync'd every 10 examples. On start the runner reads the file, collects finished uids, and skips them. The wall-clock guard stops at an example boundary while there is still time to write the manifest and print the resume command. So: **the start command and the resume command are the same command.** The only thing you must do between sessions is carry the run directory forward (cell 11).

## 1. Detect the hardware you were actually given

Kaggle sometimes hands you a P100 when you asked for T4x2. Check every session: it changes batch size, dtype and whether sharding is available.

In [ ]:
import os, shutil, subprocess, sys

print(sys.version)
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "NO GPU VISIBLE")
for path in ("/kaggle/working", "/kaggle/temp", "/tmp"):
    if os.path.isdir(path):
        total, used, free = shutil.disk_usage(path)
        print(f"{path:16s} free={free/1024**3:6.1f} GB  total={total/1024**3:6.1f} GB")
print("IS_KAGGLE:", os.path.isdir("/kaggle"), "| run type:", os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))

## 2. Get the harness code

Three options, tried in order:
1. Already present in the working directory (you copied it in).
2. Attached as a **Kaggle Dataset** input (works with internet OFF) - upload the repo as a dataset once, attach it every session.
3. `git clone` (needs internet ON). Set `REPO_URL` first.

In [ ]:
import glob, os, shutil, subprocess, sys
from pathlib import Path

REPO_URL = ""  # e.g. "https://github.com/you/reasoning-harness.git" (internet ON only)
WORKDIR = Path("/kaggle/working") if os.path.isdir("/kaggle/working") else Path.cwd()
HARNESS = None

# 1. already here?
for cand in (Path.cwd(), WORKDIR / "harness", WORKDIR / "reasoning-harness"):
    if (cand / "src" / "runner.py").exists():
        HARNESS = cand
        break

# 2. attached as a Kaggle Dataset? copy it out so we can write next to it.
if HARNESS is None:
    hits = sorted(glob.glob("/kaggle/input/*/src/runner.py")) + sorted(
        glob.glob("/kaggle/input/*/*/src/runner.py")
    )
    if hits:
        src_root = Path(hits[0]).parents[1]
        HARNESS = WORKDIR / "harness"
        shutil.copytree(src_root, HARNESS, dirs_exist_ok=True)
        print(f"copied harness from {src_root}")

# 3. clone
if HARNESS is None and REPO_URL:
    HARNESS = WORKDIR / "harness"
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(HARNESS)], check=True)

assert HARNESS is not None, (
    "harness not found: set REPO_URL, or attach the repo as a Kaggle Dataset input"
)
os.chdir(HARNESS)
if str(HARNESS) not in sys.path:
    sys.path.insert(0, str(HARNESS))
print("harness:", HARNESS)
print(sorted(p.name for p in HARNESS.iterdir()))

## 3. Install only what is missing

**Do not** `pip install -U torch numpy transformers`. Kaggle's torch is compiled against its numpy; replacing either gives you `numpy.dtype size changed, may indicate binary incompatibility` and a wasted session. Install the few packages Kaggle does not ship and nothing else.

In [ ]:
# Safe additions only. `math-verify` improves math grading; `bitsandbytes` is
# needed ONLY for 4-bit. Both are absent from the Kaggle image.
%pip install -q "pyyaml>=6" "math-verify>=0.5.2" "statsmodels>=0.14"

INSTALL_4BIT = False   # set True only if you need 4-bit NF4 (a 7B in fp16 fits a T4 without it)
INSTALL_VLLM = False   # optional throughput path; large install, can conflict with Kaggle's torch

if INSTALL_4BIT:
    %pip install -q "bitsandbytes>=0.50.0"
if INSTALL_VLLM:
    %pip install -q vllm

## 4. Verify the environment

Prints exactly what the paper needs to report, and warns about anything that will bite later (bf16 on Turing, missing bitsandbytes, absent vLLM).

In [ ]:
import importlib, json

for mod in ("torch", "transformers", "datasets", "accelerate", "huggingface_hub",
            "numpy", "matplotlib", "sympy", "math_verify", "bitsandbytes", "vllm"):
    try:
        m = importlib.import_module(mod)
        print(f"{mod:18s} {getattr(m, '__version__', '?')}")
    except Exception as exc:
        print(f"{mod:18s} NOT AVAILABLE ({type(exc).__name__})")

from src.models import bitsandbytes_available, log_hardware, vllm_available

hw = log_hardware()
print(json.dumps(hw.__dict__, indent=2, default=str))
print("vLLM usable      :", vllm_available(), "(HF transformers is the default either way)")
print("bitsandbytes     :", bitsandbytes_available())
if not hw.supports_bf16:
    print("NOTE: this GPU has no bf16 support -> dtype float16 (handled automatically).")

from src.answers import grader_backend_name
print("math grader      :", grader_backend_name())

## 5. Optional: offline assets (internet OFF sessions)

First, once, in a session **with** internet:

```bash
python scripts/prefetch_assets.py \
    --model Qwen/Qwen2.5-7B-Instruct --revision main \
    --datasets gsm8k math500 arc_challenge \
    --out /kaggle/working/assets --zip
```

Download `assets_bundle.zip`, upload it as a Kaggle Dataset, then attach it in later sessions and run the cell below. GPQA additionally needs an accepted licence and `HF_TOKEN` (Add-ons -> Secrets).

In [ ]:
import glob, os

USE_OFFLINE_ASSETS = False   # flip to True when you have an assets dataset attached
MODEL_LOCAL_PATH = ""        # e.g. /kaggle/input/my-assets/models/Qwen2.5-7B-Instruct
DATA_LOCAL_DIR = ""          # e.g. /kaggle/input/my-assets/datasets/gsm8k

if USE_OFFLINE_ASSETS:
    os.environ["HF_HUB_OFFLINE"] = "1"
    os.environ["HF_DATASETS_OFFLINE"] = "1"
    os.environ["TRANSFORMERS_OFFLINE"] = "1"
    if not MODEL_LOCAL_PATH:
        hits = sorted(glob.glob("/kaggle/input/*/models/*"))
        MODEL_LOCAL_PATH = hits[0] if hits else ""
    print("offline mode ON")
    print("  model :", MODEL_LOCAL_PATH or "NOT FOUND")
    print("  data  :", DATA_LOCAL_DIR or "(will resolve from the HF cache)")
else:
    for var in ("HF_HUB_OFFLINE", "HF_DATASETS_OFFLINE", "TRANSFORMERS_OFFLINE"):
        os.environ.pop(var, None)
    print("online mode (weights and datasets will be downloaded)")

# Keep the HF cache OUT of /kaggle/working: that directory is capped at 20 GB
# and is uploaded as notebook output at the end of the session.
os.environ.setdefault("HF_HOME", "/kaggle/temp/hf")
print("HF_HOME:", os.environ["HF_HOME"])

## 6. CPU validation gate (accelerator OFF, ~30 seconds)

**Confirm Settings → Accelerator is None before running this section.**

1. Mock smoke test (~10 s): exercises dataset load, prompting, JSONL append, resume, grading.
2. Unit tests (~20 s): includes GLMM primary estimator and parametric-bootstrap null calibration.

**If either fails, stop — do not enable the accelerator or spend GPU minutes.**

In [ ]:
!python -m src.runner --config configs/smoke_mock.yaml --set runtime.out_dir=/kaggle/working/results_smoke

In [ ]:
# Unit tests: GLMM + parametric null must pass before any GPU run.
!python -m pytest tests/test_glmm.py tests/test_cdv.py tests/test_variance.py -q

## 7. Optional: tiny real model (validates the transformers path)

A 135M model checks chat templating, left padding, token accounting and stop sequences against a *real* tokenizer for a rounding error's worth of quota. Worth it before loading a 7B.

In [ ]:
!python -m src.runner --config configs/smoke_tiny_model.yaml \
    --set runtime.out_dir=/kaggle/working/results_smoke \
    --set model.dtype=float16 --set runtime.max_examples=4

## 8. THE EXPERIMENT CELL (enable accelerator here)

**Only now:** Settings → Accelerator → GPU T4 x2 (or P100). Cells 1–7 must be green first.

One config = one experiment cell = one row of the results table. Edit the constants, run, and leave it alone.

`/kaggle/working` is capped at **20 GB** and is uploaded at session end — keep HF cache in `/kaggle/temp`, zip results, and do not leave model weights in `/kaggle/working`.

Before you run: **estimate the cost.** The planner below tells you whether this fits in one session.

It runs as a subprocess on purpose: a CUDA OOM kills the child, not your kernel, and the printed resume command is exact.

In [ ]:
from src.budget import estimate_run_hours
from src.models import estimate_model_vram_gb

CONFIG = "configs/gsm8k_cot_zeroshot.yaml"
MODEL = "Qwen/Qwen2.5-7B-Instruct"
OUT_DIR = "/kaggle/working/results"
TIME_BUDGET_HOURS = 8.0     # < 12 h session limit, leaves room for install + shutdown
SEED = 0
SUBSAMPLE = 200
BATCH_SIZE = 8
DTYPE = "float16"           # T4/P100 have no bf16
LOAD_IN_4BIT = False
DEVICE_MAP = "auto"         # shards across 2x T4 when both are visible
BACKEND = "hf"              # "hf" (default, safe) | "vllm" | "auto"

# --- sanity check the plan before spending quota -------------------------
vram = estimate_model_vram_gb(7.0, DTYPE, LOAD_IN_4BIT)
print(f"estimated weights VRAM: {vram:.1f} GB (plus KV cache and activations)")
for s_per_ex in (2, 5, 10, 20):
    h = estimate_run_hours(SUBSAMPLE, s_per_ex)
    fits = "fits" if h <= TIME_BUDGET_HOURS else f"NEEDS {h / TIME_BUDGET_HOURS:.1f} SESSIONS"
    print(f"  at {s_per_ex:2d} s/example -> {h:5.2f} h  ({fits})")
print("\nMeasure the real s/example in the first minutes of the run and come back to this.")

In [ ]:
# This same command is both START and RESUME. Run it again next session.
extra = []
if LOAD_IN_4BIT:
    extra += ["--set", "model.load_in_4bit=true"]
if USE_OFFLINE_ASSETS and MODEL_LOCAL_PATH:
    extra += ["--set", f"model.local_path={MODEL_LOCAL_PATH}"]
if USE_OFFLINE_ASSETS and DATA_LOCAL_DIR:
    extra += ["--set", f"data.local_dir={DATA_LOCAL_DIR}"]

cmd = [
    sys.executable, "-m", "src.runner",
    "--config", CONFIG,
    "--model", MODEL,
    "--backend", BACKEND,
    "--out-dir", OUT_DIR,
    "--seed", str(SEED),
    "--subsample", str(SUBSAMPLE),
    "--batch-size", str(BATCH_SIZE),
    "--time-budget-hours", str(TIME_BUDGET_HOURS),
    "--set", f"model.dtype={DTYPE}",
    "--set", f"model.device_map={DEVICE_MAP}",
] + extra

print(" ".join(cmd), "\n")
# Stream output live so you can watch s/example and the ETA.
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
rc = proc.wait()
print(f"\nexit code {rc}  (0 = run complete, 2 = stopped early and resumable)")

### 8b. Optional: several cells on one model load

Loading a 7B costs minutes of quota. If you want to compare strategies in one session, load once and reuse the backend. Trade-off: an OOM now kills the kernel, so run cell 8 first to confirm the model fits.

In [ ]:
RUN_MULTI = False

if RUN_MULTI:
    from src.config import load_config
    from src.models import build_backend
    from src.runner import run

    STRATEGIES = ["cot_zeroshot", "self_consistency", "best_of_n"]
    shared = [f"model.name_or_path={MODEL}", f"model.dtype={DTYPE}",
              f"model.device_map={DEVICE_MAP}", f"runtime.out_dir={OUT_DIR}",
              f"data.subsample={SUBSAMPLE}", f"runtime.seed={SEED}"]
    backend = build_backend(load_config(CONFIG, shared))
    try:
        # Split the session budget across the cells, newest first.
        per_cell = TIME_BUDGET_HOURS / len(STRATEGIES)
        for name in STRATEGIES:
            cfg = load_config(CONFIG, shared + [f"strategy.name={name}",
                                                f"runtime.time_budget_hours={per_cell}"])
            summary = run(cfg, config_path=CONFIG, backend=backend)
            if not summary.is_complete:
                print(f"{name} incomplete; resume next session before starting new cells")
                break
    finally:
        backend.close()

## 9. Grading (separate, cached, free)

Grading never needs the GPU. If you improve the answer checker, re-run this - not cell 8. This is the single most quota-saving design decision in the harness.

In [ ]:
!python -m src.grading --results-root /kaggle/working/results
# Add --force after changing src/answers.py to regrade everything from the raw traces.

## 10. Figures and tables

Reads the JSONL, writes PDF figures and `booktabs` LaTeX tables you can `\input{}` straight into the paper. Nothing here touches the GPU, so iterate freely.

In [ ]:
!python -m src.analysis.aggregate --results-dir /kaggle/working/results \
    --out-dir /kaggle/working/paper_assets --figures --tables --baseline cot_zeroshot

In [ ]:
from pathlib import Path

for p in sorted(Path("/kaggle/working/paper_assets").rglob("*")):
    if p.is_file():
        print(f"{p.stat().st_size/1024:8.1f} KB  {p}")

## 11. Carry results to the next session (do not skip this)

`/kaggle/working` is saved as notebook output, but the reliable, explicit way to continue a multi-session run is:

1. Run the cell below to zip `results/` (raw JSONL + manifests, usually a few MB).
2. Download `results_bundle.zip` from the notebook output pane.
3. **Next session:** upload that zip as a Kaggle Dataset (or add this notebook's output as an input), attach it, and unzip it into `/kaggle/working/results`.
4. Re-run cell 8 **unchanged**. Completed examples are skipped; the run continues where it stopped.

The run directory name contains the config hash, so a resumed run can only ever append to the matching experiment. Changing a semantic setting (model, dtype, sampling params, subsample, strategy params) deliberately creates a *new* directory instead of corrupting the old one.

In [ ]:
import shutil
from pathlib import Path

RESULTS = Path("/kaggle/working/results")
if RESULTS.exists():
    bundle = shutil.make_archive("/kaggle/working/results_bundle", "zip", root_dir=str(RESULTS))
    print(f"{bundle}  {Path(bundle).stat().st_size/1024**2:.2f} MB")
else:
    print("no results directory yet")

# Cumulative progress and elapsed GPU time for every run, from the manifests.
import json

total_h = 0.0
for m in sorted(RESULTS.rglob("manifest.json")) if RESULTS.exists() else []:
    st = json.loads(m.read_text())
    total_h += st.get("gpu_hours", 0.0)
    print(f"{st.get('status'):22s} {st.get('n_completed')}/{st.get('n_total'):<5} "
          f"{st.get('gpu_hours', 0):5.2f} h  {st.get('run_name')}")
    if st.get("n_completed", 0) < st.get("n_total", 0):
        print(f"    RESUME: {st.get('resume_command')}")
print(f"\ncumulative GPU-hours across these runs: {total_h:.2f} / ~30 per week")

In [ ]:
# Housekeeping: /kaggle/working is capped at 20 GB and is uploaded at session end.
# Make sure no model weights or HF cache ended up in it.
!du -sh /kaggle/working/* 2>/dev/null | sort -h | tail -20